# xView Pipeline (Colab)

This notebook mirrors the VisDrone Colab flow:
1. Mount Google Drive.
2. Copy dataset into `MyDrive` (idempotent).
3. Optionally extract ZIP archives if needed.
4. Build runtime config.
5. Run smoke pipeline.
6. Optionally run training/eval.


In [ ]:
# Step 0: install dependencies in Colab runtime.
%pip install -q ultralytics pycocotools albumentations pyyaml tqdm


In [ ]:
# Step 1: mount Google Drive and define paths.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = Path('/content/small_objects_auto_aug')
DRIVE_ROOT = Path('/content/drive/MyDrive')

DATASET_NAME = 'xView'
TARGET_DATASET_ROOT = DRIVE_ROOT / 'datasets' / 'xview_yolo'
TARGET_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)

# Source can be:
# - prepared YOLO dataset folder in Drive (recommended)
# - zip archives folder in Drive (optional unzip step below)
SOURCE_DATASET_ROOT = DRIVE_ROOT / 'datasets' / 'incoming' / 'xview_yolo'
SOURCE_ZIP_DIR = DRIVE_ROOT / 'datasets' / f'{DATASET_NAME.lower()}_zips'

print('TARGET_DATASET_ROOT =', TARGET_DATASET_ROOT)
print('SOURCE_DATASET_ROOT =', SOURCE_DATASET_ROOT)
print('SOURCE_ZIP_DIR =', SOURCE_ZIP_DIR)


In [ ]:
# Step 2: clone project repo if needed.
import subprocess
import sys

REPO_URL = 'https://github.com/s44w/small_objects_auto_aug.git'
if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Using PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
# Step 3: copy prepared dataset to MyDrive target (idempotent and safe).
import shutil

FORCE_RECOPY = False
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}


def _count_images(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def _is_ready_yolo(root: Path) -> bool:
    req = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    if not all(p.exists() for p in req):
        return False
    return _count_images(root / 'images' / 'train') > 0 and _count_images(root / 'images' / 'val') > 0


src_resolved = SOURCE_DATASET_ROOT.resolve() if SOURCE_DATASET_ROOT.exists() else SOURCE_DATASET_ROOT
dst_resolved = TARGET_DATASET_ROOT.resolve() if TARGET_DATASET_ROOT.exists() else TARGET_DATASET_ROOT

if _is_ready_yolo(TARGET_DATASET_ROOT) and not FORCE_RECOPY:
    print('[SKIP] Target dataset already exists and looks ready.')
elif src_resolved == dst_resolved:
    # Source and target point to the same directory. Do not delete/copy over itself.
    if not _is_ready_yolo(TARGET_DATASET_ROOT):
        raise FileNotFoundError(
            f'SOURCE_DATASET_ROOT == TARGET_DATASET_ROOT but dataset is not ready: {TARGET_DATASET_ROOT}\n'
            'Either place a prepared YOLO dataset there, or run ZIP extraction step.'
        )
    print('[SKIP] Source and target are the same folder; dataset already in place.')
else:
    if not SOURCE_DATASET_ROOT.exists():
        raise FileNotFoundError(
            f'SOURCE_DATASET_ROOT not found: {SOURCE_DATASET_ROOT}\n'
            'Place prepared YOLO dataset there or use Step 4 to unpack ZIPs.'
        )

    if TARGET_DATASET_ROOT.exists():
        shutil.rmtree(TARGET_DATASET_ROOT)
    shutil.copytree(SOURCE_DATASET_ROOT, TARGET_DATASET_ROOT)
    print('[OK] Dataset copied to:', TARGET_DATASET_ROOT)

print('train images:', _count_images(TARGET_DATASET_ROOT / 'images' / 'train'))
print('val images:', _count_images(TARGET_DATASET_ROOT / 'images' / 'val'))


In [ ]:
# Step 4 (optional): unpack ZIP dataset in place (if source is ZIPs).
# Use this step when SOURCE_ZIP_DIR contains archives and SOURCE_DATASET_ROOT is absent.
import zipfile
import shutil

RUN_UNZIP_FROM_SOURCE_ZIPS = False
FORCE_UNZIP = False

if RUN_UNZIP_FROM_SOURCE_ZIPS:
    if _is_ready_yolo(TARGET_DATASET_ROOT) and not FORCE_UNZIP:
        print('[SKIP] Target dataset already prepared. Set FORCE_UNZIP=True to rebuild from ZIPs.')
    else:
        if not SOURCE_ZIP_DIR.exists():
            raise FileNotFoundError(f'ZIP source dir not found: {SOURCE_ZIP_DIR}')

        zip_files = sorted(SOURCE_ZIP_DIR.glob('*.zip'))
        if not zip_files:
            raise FileNotFoundError(f'No .zip files in: {SOURCE_ZIP_DIR}')

        if TARGET_DATASET_ROOT.exists() and FORCE_UNZIP:
            shutil.rmtree(TARGET_DATASET_ROOT)
        TARGET_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

        for z in zip_files:
            print('[UNZIP]', z.name)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(TARGET_DATASET_ROOT)

        print('[OK] Unzip finished:', TARGET_DATASET_ROOT)
else:
    print('ZIP extraction skipped. Set RUN_UNZIP_FROM_SOURCE_ZIPS=True if needed.')


In [ ]:
# Step 5: build runtime config from template.
import yaml

base_cfg_path = PROJECT_ROOT / 'configs' / 'xview_config.yaml'
runtime_cfg_path = PROJECT_ROOT / 'artifacts' / f'{DATASET_NAME.lower()}_runtime_config.yaml'
runtime_cfg_path.parent.mkdir(parents=True, exist_ok=True)

with base_cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['kind'] = 'yolo_generic'
cfg['dataset']['mode'] = 'manual'
cfg['dataset']['root'] = str(TARGET_DATASET_ROOT)

# Optional only: pipeline writes its own runtime data.yaml for training.
if isinstance(cfg.get('training'), dict):
    cfg['training']['data_yaml'] = str(TARGET_DATASET_ROOT / 'dataset.yaml')

with runtime_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print('Runtime config:', runtime_cfg_path)


In [ ]:
# Step 6: smoke run (subset -> analyze -> policy).
from src.data.subset_builder import build_yolo_subset
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.utils.io import load_yaml

cfg = load_yaml(runtime_cfg_path)

subset_root = PROJECT_ROOT / 'datasets' / f'{DATASET_NAME.lower()}_smoke'
smoke_stats_dir = PROJECT_ROOT / 'artifacts' / f'{DATASET_NAME.lower()}_smoke' / 'stats'
smoke_policy_dir = PROJECT_ROOT / 'artifacts' / f'{DATASET_NAME.lower()}_smoke' / 'policy'

subset_report = build_yolo_subset(
    dataset_root=TARGET_DATASET_ROOT,
    output_root=subset_root,
    train_images=120,
    val_images=40,
    seed=42,
    clean_output=True,
)
print('subset report:', subset_report)

stats = analyze_dataset(
    dataset_root=subset_root,
    output_dir=smoke_stats_dir,
    splits=('train',),
    config=DatasetAnalyzerConfig(
        small_max_area=float(cfg['analysis']['small_max_area']),
        medium_max_area=float(cfg['analysis']['medium_max_area']),
        tiny_max_area=float(cfg['analysis']['tiny_max_area']),
        image_extensions=tuple(cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
        generate_plots=False,
    ),
)

policy, decision_report = generate_policy_from_stats(
    stats,
    cfg=RuleEngineConfig.from_project_config(cfg),
)
paths = save_policy_artifacts(policy, decision_report, output_dir=smoke_policy_dir)

print('[OK] smoke done')
print('stats:', smoke_stats_dir / 'dataset_stats.json')
print('policy:', paths['policy_json'])


In [ ]:
# Step 7: optional full pipeline run.
from src.pipeline_mvp import run_mvp_pipeline

RUN_TRAINING = False
RUN_EVAL = False
TRAIN_PROFILE = 'fast'  # fast | final | balanced | quality | hour | max_quality

outputs = run_mvp_pipeline(
    project_config_path=runtime_cfg_path,
    run_training=RUN_TRAINING,
    run_eval=RUN_EVAL,
    train_profile=TRAIN_PROFILE,
    auto_predict_for_eval=True,
)
print(outputs)
